# DOS corruption vs MSE

훈련된 DOS 체크포인트와 openwebtext 단일 샘플을 불러와서, 시간 축 t에 따라 DOS corruption → 모델 추론 → 예측 확률과 GT 원핫 간 MSE를 그립니다.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

# project imports
sys.path.append('/dataset/david/discrete-mean-flow/text')
import dataloader  # noqa: E402
import algo       # noqa: E402
from main import _load_from_checkpoint  # noqa: E402

PROJECT_ROOT = '/dataset/david/discrete-mean-flow/text'
# DOS checkpoint trained on openwebtext-20 (flow_ratio mix of FM/MF)
CHECKPOINT_PATH = (
    '/dataset/david/discrete-mean-flow/text/outputs/openwebtext-20/'
    '2025.12.02/124916/checkpoints/last.ckpt'
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Compose config for loading the DOS checkpoint.
# We override to run on a single device and to point at the chosen checkpoint.
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

accelerator_override = 'cuda' if torch.cuda.is_available() else 'cpu'

with initialize_config_dir(version_base=None, config_dir=os.path.join(PROJECT_ROOT, 'configs')):
    overrides = [
        'mode=sample_eval',
        'algo=dos',
        'data=openwebtext-1sample',  # use cached single-sample data + tokenizer
        f'eval.checkpoint_path={CHECKPOINT_PATH}',
        f'trainer.accelerator={accelerator_override}',
        'trainer.devices=1',
        'trainer.num_nodes=1',
        'trainer.strategy=null',
        'trainer.accumulate_grad_batches=1',
        'training.use_torch_compile=False',
        'training.loss_precision=float32',
        'sampling.use_float64=False',
    ]
    config = compose(config_name='config', overrides=overrides)

# Build tokenizer and load model from checkpoint
print('Tokenizing with', config.data.tokenizer_name_or_path)
tokenizer = dataloader.get_tokenizer(config)

model = _load_from_checkpoint(diffusion_model=algo.DOS, config=config, tokenizer=tokenizer)
model.eval()
model.to(device)
print('Loaded model; vocab size:', model.vocab_size)

# Disable EMA if requested in the config
if getattr(config.eval, 'disable_ema', False):
    model.ema = None

In [ ]:
# Load the openwebtext single-sample tokens (shape: [N, L]).
sample_path = os.path.join(PROJECT_ROOT, 'owt_1sample_data', 'x0.npy')
x0_np = np.load(sample_path)
print('x0.npy shape:', x0_np.shape)

# Take the first sample for analysis
x0_tokens = torch.tensor(x0_np[0], device=device, dtype=torch.long).unsqueeze(0)
print('Chosen sample shape:', x0_tokens.shape)

# Quick peek at the decoded text (truncated)
preview = tokenizer.decode(x0_tokens[0].tolist()[:256])
print('Decoded preview:\n', preview)

In [ ]:
@torch.no_grad()
def dos_mse_over_t(model, x0_tokens, t_grid):
    """
    For each alpha-space t in t_grid [0,1], corrupt the sample following DOS
    training (continuous corruption with gamma schedule), run the model, and
    return mean MSE (averaged over tokens and vocab).
    """
    mse_vals = []
    for t_val in t_grid:
        t_alpha = torch.tensor([float(t_val)], device=device)
        if model.config.algo.use_discrete_schedule:
            gamma_t = model._alpha_t_to_gamma(t_alpha)
        else:
            gamma_t = t_alpha

        x_t, target = model.corrupt_continuous(x0_tokens, gamma_t)
        if model.config.algo.use_curriculum:
            x_t = x_t * model._compute_gumbel_tau_inverse()

        pred_log = model.forward(x_t, gamma_t)
        pred = pred_log.exp()
        mse = ((pred - target) ** 2).mean(dim=-1).mean()
        mse_vals.append(mse.item())
    return mse_vals

# Evaluate across a grid of t values
num_points = 25
t_grid = np.linspace(0.0, 1.0, num=num_points)
mse_vals = dos_mse_over_t(model, x0_tokens, t_grid)

print('MSE stats -> min:', np.min(mse_vals), 'max:', np.max(mse_vals))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t_grid, mse_vals, marker='o')
plt.xlabel('t (alpha space)')
plt.ylabel('MSE (pred vs GT one-hot)')
plt.title('DOS corruption → prediction error over t')
plt.grid(True)
plt.show()

# Optional log-scale view for variance-heavy runs
plt.figure(figsize=(8, 4))
plt.semilogy(t_grid, mse_vals, marker='o')
plt.xlabel('t (alpha space)')
plt.ylabel('MSE (log scale)')
plt.title('DOS corruption → prediction error (log scale)')
plt.grid(True, which='both')
plt.show()